# 04 - Search Methods: Embeddings

This notebook builds the second search method, embedding search, and scores it on the same 30 dev questions as all search methods that we try. The question stays as it is in the dataset, no rewriting, so the comparison with BM25 is a fair one.

Previous search method, BM25, matches exact words so it cannot connect "PPNE" to "Property, plant and equipment", and it overweights rare instruction words such as "analyst". Embedding search compares meaning instead of exact words, which is supposed to help with both.

This notebook runs on Google Colab (because of embeddings).

In [1]:
!git clone https://github.com/dogamentese/financebench-rag.git
%cd financebench-rag
!pip install -q -U sentence-transformers qdrant-client
!pip install -q pymupdf bm25s
!pip install -q -e .
!python scripts/download_data.py

Cloning into 'financebench-rag'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 39 (delta 6), reused 34 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 636.45 KiB | 12.73 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/financebench-rag
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for financebench-rag (pyproject.toml) ... done
[1/84] 3M_2018_10K
[2/84] 3M_2022_10K
[3/84] 3M_2023Q2_10Q
[4/84] ACTIVISIONBLIZZARD_2019_10K
[5/84] ADOBE_2015_10K
[6/84] ADOBE_2016_10K
[7/84] ADOBE_2017_10K
[8/84] ADOBE_2022_10K
[9/84] AES_2022_10K
[10/84] AMAZON_2017_10K
[11/84] AMAZON_2019_10K
[12/84] AMCOR_2020_10K
[13/84] AMCOR_2022_8K_dated-2022-07-01
[14/84] AMCOR_2023Q2_10Q
[15/84] AMCOR_2023Q4

## 1. Load the embedding model

An embedding model turns a text into a vector, a list of numbers that represents its meaning. Texts with similar meaning get vectors that point in a similar direction, even when they share no words. Therefore our steps are:
- embedding the question
- embedding every chunk
- returning the chunks whose vectors are closest to the question's vector

The model is Qwen3-Embedding-0.6B, an open-weights embedding model. It embeds questions and chunks differently: a question is encoded with `prompt_name="query"`, which prepends an instruction the model was trained with, while chunks are encoded as they are.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", model_kwargs={"dtype": "float16"})
model.max_seq_length = 512
print("vector size:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vector size: 1024


/tmp/ipykernel_2334/2712460331.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("vector size:", model.get_sentence_embedding_dimension())


## 2. Prepare the chunks

The chunks are built exactly as in notebook 03: 250 words, 50 words overlap, and the prefix with company, document type and year.

In [ ]:
import sys
sys.path.append("/content/financebench-rag/src")

In [6]:
from financebench_rag.data import REPO, extract_pages, gold_pages, load_doc_info, load_pages, load_questions, load_split
from financebench_rag.retrieval import build_chunks

questions = load_questions()
dev_ids = set(load_split()["dev"])
dev = [q for q in questions if q["financebench_id"] in dev_ids]

extract_pages()
pages = load_pages()
chunks = build_chunks(pages, load_doc_info(), size=250, overlap=50, prefix=True)
print(f"{len(chunks)} chunks from {len(pages)} pages")

33998 chunks from 12013 pages


## 3. Embed the chunks and index them in Qdrant

The cell below embeds every chunk once and stores the vectors in Qdrant, a vector database. Qdrant's job is the "closest vectors" lookup at search time.

In [7]:
import time

from financebench_rag.embedding import EmbeddingSearch

start = time.time()
search = EmbeddingSearch(chunks, model)
print(f"embedded and indexed in {(time.time() - start) / 60:.1f} min")

Batches:   0%|          | 0/532 [00:00<?, ?it/s]

/content/financebench-rag/src/financebench_rag/embedding.py:18: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert("chunks", points)


embedded and indexed in 27.1 min


## 4. Two example searches

The same two questions as in notebook 03.

In [8]:
for fid in ["financebench_id_01482", "financebench_id_04672"]:
    q = next(q for q in dev if q["financebench_id"] == fid)
    print("Q:", q["question"])
    print("evidence page:", sorted(gold_pages(q)))
    print("top 5 pages:")
    for page in search.search(q["question"], k=5):
        print("  ", page)
    print()

Q: At the Pepsico AGM held on May 3, 2023, what was the outcome of the shareholder vote on the shareholder proposal for a congruency report by Pepsico on net-zero emissions policies?
evidence page: [('PEPSICO_2023_8K_dated-2023-05-05', 3)]
top 5 pages:
   ('PEPSICO_2023_8K_dated-2023-05-05', 3)
   ('PEPSICO_2023_8K_dated-2023-05-05', 2)
   ('PEPSICO_2022_10K', 118)
   ('PEPSICO_2022_10K', 132)
   ('PEPSICO_2022_10K', 134)

Q: Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
evidence page: [('3M_2018_10K', 57)]
top 5 pages:
   ('3M_2018_10K', 59)
   ('3M_2018_10K', 55)
   ('3M_2018_10K', 13)
   ('3M_2018_10K', 15)
   ('3M_2018_10K', 126)



Both questions behave differently than in notebook 03:

- The PepsiCo question is still found at rank 1, so the embedding search keeps the one BM25 already got right. The rest of its top 5 is other PepsiCo filings rather than the unrelated reports BM25 returned.
- The 3M PPNE question is at rank 6 instead of 1304th. All of its top 5 pages come from the right report, so the model connects "PPNE" to "Property, plant and equipment" well enough to find the 3M 2018 10-K, and the remaining difficulty is picking the right page inside it.

## 5. Scores on the dev questions

The metrics are the ones from notebook 03: recall@1, recall@5, mrr@10 and doc_recall@5, a hit being any evidence page of the question. The BM25 row is loaded from `results/dev_retrieval_bm25.json` so both methods sit in one table.

In [9]:
import json

import pandas as pd

from financebench_rag.evaluate import evaluate_retrieval

results = {q["financebench_id"]: search.search(q["question"], k=10) for q in dev}

bm25_results = {
    fid: [tuple(page) for page in ranked]
    for fid, ranked in json.load(open(REPO / "results" / "dev_retrieval_bm25.json")).items()
}

pd.DataFrame([
    {"method": "BM25", **evaluate_retrieval(bm25_results, dev)},
    {"method": "embeddings", **evaluate_retrieval(results, dev)},
])

,method,recall@1,recall@5,mrr@10,doc_recall@5
0,BM25,0.100,0.133,0.111,0.6
1,embeddings,0.233,0.500,0.351,1.0


In [10]:
# The same recall@5, split by question type.
hits = pd.DataFrame({
    "question_type": [q["question_type"] for q in dev],
    "hit@5": [any(page in gold_pages(q) for page in results[q["financebench_id"]][:5]) for q in dev],
})
hits.groupby("question_type")["hit@5"].agg(["sum", "count"])

,sum,count
question_type,,
domain-relevant,3,10
metrics-generated,7,10
novel-generated,5,10


In [11]:
# The dev questions whose evidence page is in the top 10, with the rank of the evidence page.
pd.set_option("display.max_colwidth", None)
found = []
for q in dev:
    ranked = results[q["financebench_id"]]
    rank = next((i + 1 for i, page in enumerate(ranked) if page in gold_pages(q)), None)
    if rank:
        found.append({"rank": rank, "question_type": q["question_type"], "question": q["question"]})
pd.DataFrame(found)

,rank,question_type,question
0,6,metrics-generated,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
1,4,novel-generated,Does 3M maintain a stable trend of dividend distribution?
2,1,metrics-generated,What is the FY2019 fixed asset turnover ratio for Activision Blizzard? Fixed asset turnover ratio is defined as: FY2019 revenue / (average PP&E between FY2018 and FY2019). Round your answer to two decimal places. Base your judgments on the information provided primarily in the statement of income and the statement of financial position.
3,3,metrics-generated,What is the FY2017 operating cash flow ratio for Adobe? Operating cash flow ratio is defined as: cash from operations / total current liabilities. Round your answer to two decimal places. Please utilize information provided primarily within the balance sheet and the cash flow statement.
4,1,domain-relevant,"Does Adobe have an improving operating margin profile as of FY2022? If operating margin is not a useful metric for a company like this, then state that and explain why."
5,3,novel-generated,Does Adobe have an improving Free cashflow conversion as of FY2022?
6,2,metrics-generated,What is Amazon's FY2017 days payable outstanding (DPO)? DPO is defined as: 365 * (average accounts payable between FY2016 and FY2017) / (FY2017 COGS + change in inventory between FY2016 and FY2017). Round your answer to two decimal places. Address the question by using the line items and information shown within the balance sheet and the P&L statement.
7,4,domain-relevant,"Does AMCOR have an improving gross margin profile as of FY2023? If gross margin is not a useful metric for a company like this, then state that and explain why."
8,6,metrics-generated,"Answer the following question as if you are an equity research analyst and have lost internet connection so you do not have access to financial metric providers. According to the details clearly outlined within the P&L statement and the statement of cash flows, what is the FY2015 depreciation and amortization (D&A from cash flow statement) % margin for AMD?"
9,1,domain-relevant,Which debt securities are registered to trade on a national securities exchange under American Express' name as of 2022?


## 6. Save the results

The top 10 pages per dev question are saved in the same format as the BM25 file and downloaded, since files on Colab disappear when the session ends.

In [12]:
out_path = REPO / "results" / "dev_retrieval_embeddings.json"
out_path.write_text(json.dumps(results) + "\n")
print("wrote", out_path)

from google.colab import files
files.download(out_path)

wrote /content/financebench-rag/results/dev_retrieval_embeddings.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary of Results

- Embeddings reach a recall@5 of 0.50 against 0.13 for BM25, and a doc_recall@5 of 1.00 against 0.60. The right report is now always in the top 5, so the remaining failures are about picking the right page inside it.
- Metrics questions, which ask for a number from a financial statement, go from 0 of 10 found in the top 5 in notebook 03 to 7 of 10 here. This was the worst category for BM25 and is the best one for embeddings.
- Two questions that BM25 found are now missed (the AMCOR 8-K filing at rank 5 and the Ulta Beauty inventory question at rank 8, can be seen in notebook03), so the two methods fail on different questions.
- Query rewriting is not tested here. The prompt-style wording hurt BM25 in notebook 03 because it matches exact words, and we expected meaning-based search to be less sensitive to it. The recall@5 of 0.50 on the unmodified questions was good enough that rewriting was not needed, though whether it would help further is an open question.



